In [1]:

import pandas as pd

DATA_PATH = r"D:\Programing\Github\Persian-Medical-RAG-Chatbot\data\raw\Specialized_Dataset.xlsx"

def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    print(f"Total number of records : {len(df)}")
    print(f"Number of columns: {len(df.columns)}")
    print(f"Name of columns: {list(df.columns)}")
    return df

def basic_report(df: pd.DataFrame) -> None:
    print("\n=== Number of empty values per column ===")
    print(df.isnull().sum())

    print("\n=== Records with answers ===")
    has_response = df["response"].notna().sum()
    print(f"{has_response} out of {len(df)} records have answer")

    print("\n=== Number of unique drugs ===")
    print(df["drug_name"].nunique())

    print("\n=== Example of a record ===")
    sample = df[df["response"].notna()].iloc[0]
    print("Drug :", sample["drug_name"])
    print("Question:", sample["comment"])
    print("Response:", sample["response"])

if __name__ == "__main__":
    df = load_dataset(DATA_PATH)
    basic_report(df)


Total number of records : 12399
Number of columns: 16
Name of columns: ['commenter_id', 'date', 'comment', 'response', 'english_translated_comment', 'english_translated_response', 'drug_name', 'martindale_english', 'persian_martidale_category', 'doctor_id', 'doctor_category', 'translated_doctor_catg', 'doctor_speciality', 'translated_doctor_speciality', 'therapeutic_category', 'translated_therapeutic_category']

=== Number of empty values per column ===
commenter_id                           0
date                                   0
comment                                0
response                            7680
english_translated_comment             0
english_translated_response         7680
drug_name                              0
martindale_english                     0
persian_martidale_category             0
doctor_id                           7680
doctor_category                     7680
translated_doctor_catg              7680
doctor_speciality                  10349
translate

In [2]:
df.shape

(12399, 16)

In [4]:
df.columns

Index(['commenter_id', 'date', 'comment', 'response',
       'english_translated_comment', 'english_translated_response',
       'drug_name', 'martindale_english', 'persian_martidale_category',
       'doctor_id', 'doctor_category', 'translated_doctor_catg',
       'doctor_speciality', 'translated_doctor_speciality',
       'therapeutic_category', 'translated_therapeutic_category'],
      dtype='str')

In [25]:
import re
import pandas as pd 
DATA_PATH = r"D:\Programing\RAG_Project's me\datasets\Dataset.xlsx"
OUTPUT_PATH = r"D:\Programing\RAG_Project's me\datasets\cleaned_dataset.csv"

ARABIC_TO_PERSIAN = {
    "ي": "ی",
    "ك": "ک",
    "ة": "ه",
    "ۀ": "ه",
    "أ": "ا",
    "إ": "ا",
    "ؤ": "و",
    "ئ": "ی",
}

# Arabic diacritics and formation signs that should be removed
ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]"
)

# Emojis and pictographic symbols
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+",
    flags=re.UNICODE,
)


def normalize_text(text: str) -> str:
    """Clean and normalize a Persian text string."""
    if not isinstance(text, str):
        return ""

    # Remove remaining Excel line-break characters
    text = text.replace("_x000D_", " ")
    text = text.replace("\r", " ").replace("\n", " ")

    # Normalize Arabic characters to Persian characters
    for ar, fa in ARABIC_TO_PERSIAN.items():
        text = text.replace(ar, fa)

    # Remove Arabic diacritics
    text = ARABIC_DIACRITICS.sub("", text)

    # Remove emojis
    text = EMOJI_PATTERN.sub("", text)

    # Normalize whitespace (multiple spaces -> one space)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def load_and_clean(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    print(f"Raw record count: {len(df)}")

    # 1. Remove records without a response
    df = df[df["response"].notna()].copy()
    print(f"Records with response: {len(df)}")

    # 2. Clean question and response text
    df["comment_clean"] = df["comment"].apply(normalize_text)
    df["response_clean"] = df["response"].apply(normalize_text)

    # 3. Remove records with very short responses (less than 10 characters)
    before = len(df)
    combined_len = df["comment_clean"].str.len() + df["response_clean"].str.len()
    df = df[combined_len >= 20].copy()
    print(f"Removed (combined question+answer too short/empty): {before - len(df)}")

    # 4. Create combined text for retrieval (this text will be embedded later)
    df["retrieval_text"] = (
        "دارو: " + df["drug_name"].astype(str) + " | سؤال: "
        + df["comment_clean"] + " | پاسخ: " + df["response_clean"]
    )

    # 5. Keep only the required final columns
    final_cols = [
        "commenter_id",
        "drug_name",
        "persian_martidale_category",
        "therapeutic_category",
        "comment_clean",
        "response_clean",
        "retrieval_text",
    ]

    df_final = df[final_cols].reset_index(drop=True)
    df_final.insert(0, "doc_id", range(1, len(df_final) + 1))

    return df_final


if __name__ == "__main__":
    import os

    os.makedirs("data", exist_ok=True)

    df_clean = load_and_clean(DATA_PATH)

    print("\n=== Sample final record ===")
    print(df_clean.iloc[0]["retrieval_text"])

    df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"\nSaved to: {OUTPUT_PATH} ({len(df_clean)} records)")

Raw record count: 12399
Records with response: 4719
Removed (combined question+answer too short/empty): 1

=== Sample final record ===
دارو: Rosa-damascene | سؤال: سلام‌ تو اینترنت خوندم عصاره گل سرخ واسه افسردگی و اضطراب خوبه؟من اختلال دوقطبی دارم میخواستم بدونم آیا عصاره گل سرخ میتونه باعث القا حالت مانیا در من بشه؟ | پاسخ: عصاره گل رز تا حدودی اثرات شادی آور دارد ولی به هیچ وجه جایگزین داروهای درمان دوقطبی نیست.

Saved to: D:\Programing\RAG_Project's me\datasets\cleaned_dataset.csv (4718 records)


In [26]:
df_clean.columns

Index(['doc_id', 'commenter_id', 'drug_name', 'persian_martidale_category',
       'therapeutic_category', 'comment_clean', 'response_clean',
       'retrieval_text'],
      dtype='str')

In [27]:
df_clean.shape

(4718, 8)